In [1]:
import os
import pandas as pd
import plotly.express as px
import plotly.io as pio
import seaborn as sns
from neo4j import Record
from neo4j_viz import Node, Relationship, VisualizationGraph
from IPython.display import HTML

In [2]:
pio.templates.default = "seaborn"

In [3]:
path = "/home/abdelnour-main/Documents/Projects/Text2KG/data/nantes-uni-2020/expertise_map"

In [4]:
def str2array(s : str) -> list[str]:

    if s == '[]':
        return []

    s = s[1:-1]

    return [
        domain.strip()[1:-1]
        for domain in s.split(',')
    ]

In [5]:
nodes_metadata = pd.read_csv(os.path.join(path, 'nodes_metadata.csv'))

community_metadata = pd.read_csv(os.path.join(path, 'community_metadata.csv'))
community_metadata['domains'] = community_metadata['domains'].apply(str2array)
community_metadata['candidates'] = community_metadata['candidates'].apply(str2array)
community_metadata['size'] = community_metadata['candidates'].apply(len)
mask = (community_metadata['size'] >= 30) & (community_metadata['domains'].apply(len).gt(0))
community_metadata = community_metadata[mask]

In [6]:
for label in nodes_metadata.columns[1:19]:
    community_metadata[label] = community_metadata['domains'].apply(lambda x : label in x)

In [25]:
from openai import Client

client = Client(
    api_key="sk-46e0b8e6cb7d4c989b13af3980c397c3",
    base_url="https://api.deepseek.com"
)

def evaluate_community(client : Client, entities : list[str], k : int = 10) -> str:

    messages = [
        {
            "role": "system",
            "content": (
                "You are a ranking system that you will be given a group of pipe ( | ) separated concepts "
                f"and you are tasked with selecting {k} concepts that encodes the domains present in the group the best. "
                f"The result should be a pipe separated list of the {k} entities ranked from the most relevant concept as the "
                "to least relevant concept (last), no additional explanation is required."
            ),
        },
        {
            "role": "user",
            "content": "|".join(entities)
        }
    ]

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        stream=False,
        temperature=0,  # No randomness
        top_p=1,        # Disable sampling
        seed=42,        # Seed
    )

    content = response.choices[0].message.content
    entities = [entity.strip(' ') for entity in content.split('|')]

    return entities

In [45]:
from tqdm.auto import tqdm

results = []

for _, row in tqdm(community_metadata.iterrows(), total=len(community_metadata)):
    result = evaluate_community(client, row['candidates'])
    results.append(row.to_dict())
    results[-1]['top-10'] = result

  0%|          | 0/303 [00:00<?, ?it/s]

In [46]:
metrics = {
    'top-1' : 0.0,
    'top-3' : 0.0,
    'top-5' : 0.0,
    'top-10' : 0.0
}

for result in results:
    metrics['top-1'] += int(result['name'] == result['top-10'][0]) / len(results)
    metrics['top-3'] += int(result['name'] in result['top-10'][:3]) / len(results)
    metrics['top-5'] += int(result['name'] in result['top-10'][:5]) / len(results)
    metrics['top-10'] += int(result['name'] in result['top-10'][:10]) / len(results)

In [47]:
metrics

{'top-1': 0.4158415841584155,
 'top-3': 0.5544554455445544,
 'top-5': 0.5907590759075911,
 'top-10': 0.6633663366336646}

In [49]:
community_metadata.to_csv('metadata_llm.csv')